In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from google import genai

def list_available_models(api_key: str = None):
    client = genai.Client(api_key=api_key)

    print(f"{'MODEL NAME':<40} | {'SUPPORTED METHODS'}")
    print("-" * 80)

    for model in client.models.list():
        name = model.name

        methods = getattr(model, 'supported_generation_methods', [])
        methods_str = ", ".join(methods) if methods else "N/A"

        print(f"{name:<40} | {methods_str}")


list_available_models()

MODEL NAME                               | SUPPORTED METHODS
--------------------------------------------------------------------------------
models/gemini-2.5-flash                  | N/A
models/gemini-2.5-pro                    | N/A
models/gemini-2.0-flash                  | N/A
models/gemini-2.0-flash-001              | N/A
models/gemini-2.0-flash-lite-001         | N/A
models/gemini-2.0-flash-lite             | N/A
models/gemini-2.5-flash-preview-tts      | N/A
models/gemini-2.5-pro-preview-tts        | N/A
models/gemma-3-1b-it                     | N/A
models/gemma-3-4b-it                     | N/A
models/gemma-3-12b-it                    | N/A
models/gemma-3-27b-it                    | N/A
models/gemma-3n-e4b-it                   | N/A
models/gemma-3n-e2b-it                   | N/A
models/gemma-4-26b-a4b-it                | N/A
models/gemma-4-31b-it                    | N/A
models/gemini-flash-latest               | N/A
models/gemini-flash-lite-latest          | N/A
models/gemin

In [3]:
from llama_index.core import Settings, StorageContext
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")

Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

In [4]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader('data').load_data()
index = VectorStoreIndex.from_documents(documents)

In [5]:
query_engine = index.as_query_engine()

In [6]:
response = await query_engine.aquery("Do I like linux?")
print(response)

Yes, you love Linux.


In [7]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.workflow import Context

def get_favorite_color():
  """A function that returns the user's favorite color."""
  return "blue"

async def get_context_from_database(query: str) -> str:
  """A function that takes in a query and returns relevant context from the database."""
  result = await query_engine.aquery(query)
  return str(result)

agent = FunctionAgent(
    llm=GoogleGenAI(model='gemini-2.5-flash'),
    tools=[get_favorite_color, get_context_from_database],
    system_prompt='You are a helpful assistant. If you do not have access to information needed for the answer, you can use the tools provided to get the information.',
)

ctx = Context(agent)

In [8]:
response = await agent.run('What is my favorite color?', ctx=ctx)
print(response)

Your favorite color is blue.


In [9]:
response = await agent.run('What is my favorite food?')
print(response)

Your favorite food is oranges.


In [10]:
index.storage_context.persist('storage')